# Scientific Article Citation Prediction with DBLP

This notebook uses the Kaggle DBLP Citation Network V14 dataset and compares k-nearest neighbors with a small neural network. It reads a manageable sample from the large archive, keeps each paper ID once, and uses the test set only for final evaluation.

In [ ]:
import ast
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Download the dataset

This cell downloads only the required compressed CSV through the Kaggle CLI. If the archive is already present, it is not downloaded again. The Kaggle package can be installed with `pip install kaggle`.

In [ ]:
from pathlib import Path
import subprocess
import sys

KAGGLE_DATASET = "agungpambudi/research-citation-network-5m-papers"
DATASET_FILE = "dblp-citation-network-v14.csv"
DATASET_DIRECTORY = Path("dataset")
DATASET_PATH = DATASET_DIRECTORY / f"{DATASET_FILE}.zip"

if DATASET_PATH.exists():
    print("Dataset already downloaded:", DATASET_PATH)
else:
    DATASET_DIRECTORY.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable, "-m", "kaggle", "datasets", "download",
            "-d", KAGGLE_DATASET,
            "-f", DATASET_FILE,
            "-p", str(DATASET_DIRECTORY),
        ],
        check=True,
    )
    print("Dataset downloaded:", DATASET_PATH)

## Load and prepare the Kaggle data

The full CSV is 3.28 GB, so only the first 250,000 records are read and a fixed sample of 100,000 suitable papers is selected. The source is filtered to English journal and conference papers published from 2000 through 2018, giving citations time to accumulate.

In [ ]:
ROWS_TO_READ = 250_000
SAMPLE_SIZE = 100_000
REFERENCE_YEAR = 2024

columns = [
    "id", "title", "keywords", "lang", "venue", "year",
    "n_citation", "abstract", "authors", "doc_type", "references"
]
raw_df = pd.read_csv(
    DATASET_PATH, sep="|", usecols=columns,
    nrows=ROWS_TO_READ, low_memory=False
)

df = raw_df.loc[
    raw_df["id"].notna()
    & raw_df["title"].notna()
    & raw_df["abstract"].notna()
    & raw_df["lang"].eq("en")
    & raw_df["doc_type"].isin(["Journal", "Conference"])
    & raw_df["year"].between(2000, 2018)
    & raw_df["n_citation"].ge(0)
].drop_duplicates(subset="id").copy()

df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=SEED)

def list_length(value):
    if pd.isna(value):
        return 0
    try:
        return len(ast.literal_eval(value))
    except (ValueError, SyntaxError):
        return 0

df["Paper Age"] = REFERENCE_YEAR - df["year"]
df["Authors Count"] = df["authors"].apply(list_length)
df["Reference Count"] = df["references"].apply(list_length)
df["Keywords Count"] = df["keywords"].apply(list_length)
df["Title Length"] = df["title"].str.split().str.len()
df["Abstract Length"] = df["abstract"].str.split().str.len()
df["Log Citations"] = np.log1p(df["n_citation"])

features = [
    "Paper Age", "Authors Count", "Reference Count",
    "Keywords Count", "Title Length", "Abstract Length"
]
categorical_features = ["venue", "doc_type"]
text_features = ["title", "abstract"]
model_columns = features + categorical_features + text_features
target = "Log Citations"

print("Rows read:       ", len(raw_df))
print("Papers selected: ", len(df))
print("Citation range:  ", int(df["n_citation"].min()), "to", int(df["n_citation"].max()))
df[features + ["n_citation", target]].head()

## Independent train, validation, and test sets

The validation set selects model settings. The test set remains untouched until the final comparison. Numerical features are standardized, venue and document type are one-hot encoded, and title and abstract text are converted with TF-IDF. Truncated SVD compresses the resulting sparse features to 25 components so both models remain lightweight. Every preprocessing step learns only from training data.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED
)
train_df, validation_df = train_test_split(
    train_df, test_size=0.25, random_state=SEED
)

X_train = train_df[model_columns]
y_train = train_df[target].to_numpy(dtype=np.float32)
X_validation = validation_df[model_columns]
y_validation = validation_df[target].to_numpy(dtype=np.float32)
X_test = test_df[model_columns]
y_test = test_df[target].to_numpy(dtype=np.float32)

preprocessor = ColumnTransformer([
    ("numbers", StandardScaler(), features),
    ("categories", OneHotEncoder(
        handle_unknown="ignore", min_frequency=100, max_categories=100
    ), categorical_features),
    ("title", TfidfVectorizer(
        max_features=1_000, min_df=5, stop_words="english"
    ), "title"),
    ("abstract", TfidfVectorizer(
        max_features=3_000, min_df=5, stop_words="english"
    ), "abstract"),
])

svd = TruncatedSVD(n_components=25, random_state=SEED)
feature_scaler = StandardScaler()

X_train_processed = preprocessor.fit_transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

X_train_scaled = feature_scaler.fit_transform(
    svd.fit_transform(X_train_processed)
).astype(np.float32)
X_validation_scaled = feature_scaler.transform(
    svd.transform(X_validation_processed)
).astype(np.float32)
X_test_scaled = feature_scaler.transform(
    svd.transform(X_test_processed)
).astype(np.float32)

print("Train:     ", X_train_scaled.shape)
print("Validation:", X_validation_scaled.shape)
print("Test:      ", X_test_scaled.shape)

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred),
    }

## k-nearest neighbors

A short validation loop selects `k`. The test set is evaluated once after selection.

In [ ]:
k_values = [5, 10, 20, 40, 80, 120]
neighbor_search = KNeighborsRegressor(
    n_neighbors=max(k_values), algorithm="kd_tree"
)
neighbor_search.fit(X_train_scaled, y_train)
validation_neighbors = neighbor_search.kneighbors(
    X_validation_scaled, return_distance=False
)

best_k = None
best_validation_rmse = np.inf

for k in k_values:
    validation_prediction = y_train[
        validation_neighbors[:, :k]
    ].mean(axis=1)
    validation_rmse = np.sqrt(
        mean_squared_error(y_validation, validation_prediction)
    )

    if validation_rmse < best_validation_rmse:
        best_k = k
        best_validation_rmse = validation_rmse

knn = KNeighborsRegressor(n_neighbors=best_k, algorithm="kd_tree")
knn.fit(X_train_scaled, y_train)
knn_prediction = knn.predict(X_test_scaled)
knn_results = evaluate(y_test, knn_prediction)

print("Best k:", best_k)
print("Validation RMSE:", best_validation_rmse)
print("Test results:", knn_results)

## Small neural network

The network has one small hidden layer. Training stops when validation loss has not improved for 40 epochs, and the best saved weights are restored before the test evaluation.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
X_validation_tensor = torch.tensor(X_validation_scaled, dtype=torch.float32).to(device)
y_validation_tensor = torch.tensor(y_validation, dtype=torch.float32).view(-1, 1).to(device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

class SimpleNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Softplus(),
        )

    def forward(self, x):
        return self.network(x)

model = SimpleNet(X_train_tensor.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses = []
validation_losses = []
best_validation_loss = np.inf
best_state = None
epochs_without_improvement = 0
patience = 40

for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    train_prediction = model(X_train_tensor)
    train_loss = criterion(train_prediction, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        validation_prediction = model(X_validation_tensor)
        validation_loss = criterion(validation_prediction, y_validation_tensor)

    train_losses.append(train_loss.item())
    validation_losses.append(validation_loss.item())

    if validation_loss.item() < best_validation_loss:
        best_validation_loss = validation_loss.item()
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement == patience:
        break

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    neural_prediction = model(X_test_tensor).cpu().numpy().ravel()

neural_results = evaluate(y_test, neural_prediction)
print("Device:", device)
print("Epochs trained:", len(train_losses))
print("Test results:", neural_results)

## Honest model comparison

The mean baseline shows whether either model improves on a simple prediction. Metrics are calculated on `log(1 + citations)`, which is more suitable for the highly skewed citation distribution.

In [ ]:
baseline_prediction = np.full(len(y_test), y_train.mean())
baseline_results = evaluate(y_test, baseline_prediction)

comparison = pd.DataFrame(
    [baseline_results, knn_results, neural_results],
    index=["Mean baseline", "kNN", "Neural network"],
)
comparison

## Five-fold cross-validation

Cross-validation evaluates kNN on five different divisions of a fixed 20,000-paper subset. This keeps the text-based validation practical to rerun. One-hot encoding, TF-IDF, dimensionality reduction, and scaling are fitted separately inside every fold, preventing validation leakage.

In [ ]:
cv_df = df.sample(n=20_000, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_model = make_pipeline(
    preprocessor,
    TruncatedSVD(n_components=25, random_state=SEED),
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=best_k, algorithm="kd_tree"),
)

cv_results = cross_validate(
    cv_model,
    cv_df[model_columns],
    cv_df[target],
    cv=cv,
    scoring={"R²": "r2", "RMSE": "neg_root_mean_squared_error"},
)

cv_r2 = cv_results["test_R²"]
cv_rmse = -cv_results["test_RMSE"]

print("R² for each fold:", cv_r2)
print("Mean R²:", cv_r2.mean())
print("R² standard deviation:", cv_r2.std())
print("Mean RMSE:", cv_rmse.mean())

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Training loss")
plt.plot(validation_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Neural Network Learning Curves")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(7, 7))
plt.scatter(y_test, knn_prediction, label="kNN")
plt.scatter(y_test, neural_prediction, label="Neural network")
limits = [y_test.min(), y_test.max()]
plt.plot(limits, limits, "r--")
plt.xlabel("Actual log(1 + citations)")
plt.ylabel("Predicted log(1 + citations)")
plt.title("Predictions for Unseen Articles")
plt.legend()
plt.show()

## Example prediction

The input uses the same named features and training-fitted scaler as the models. The logarithmic predictions are converted back to approximate citation counts. This is an estimate for papers similar to the training sample, not a guarantee of future impact.

In [ ]:
new_article = pd.DataFrame({
    "Paper Age": [6],
    "Authors Count": [4],
    "Reference Count": [30],
    "Keywords Count": [5],
    "Title Length": [12],
    "Abstract Length": [180],
    "venue": ["{'raw': 'Neural Information Processing Systems'}"],
    "doc_type": ["Conference"],
    "title": ["Efficient neural networks for scientific data analysis"],
    "abstract": ["We present an efficient neural network method for analyzing scientific data and compare it with existing machine learning approaches."],
})
new_article_processed = preprocessor.transform(new_article[model_columns])
new_article_scaled = feature_scaler.transform(
    svd.transform(new_article_processed)
).astype(np.float32)

knn_log_prediction = knn.predict(new_article_scaled)[0]
new_article_tensor = torch.tensor(
    new_article_scaled, dtype=torch.float32
).to(device)

model.eval()
with torch.no_grad():
    neural_log_prediction = model(new_article_tensor).item()

print("kNN predicted citations:", np.expm1(knn_log_prediction))
print("Neural network predicted citations:", np.expm1(neural_log_prediction))